In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from google_play_scraper import Sort, reviews
from tqdm import tqdm
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split

from utils import classify_sentiment, balance_by_min_class

In [5]:
app_ids = {
    'ChatGPT': 'com.openai.chatgpt',
    'Gemini': 'com.google.android.apps.bard',
    'Claude': 'com.anthropic.claude'
}

In [6]:
reviews_per_app = 100000
all_reviews = []

In [ ]:
for app_name, app_id in app_ids.items():
    print(f"Scraping reviews for {app_name}...")

    app_results = []
    continuation_token = None

    with tqdm(total=reviews_per_app, position=0, leave=True) as pbar:
        while len(app_results) < reviews_per_app:
            new_result, continuation_token = reviews(
                app_id,
                continuation_token=continuation_token,
                lang='en',
                country='us',
                sort=Sort.NEWEST,
                count=199,  # API limit
            )

            if not new_result:
                print(f"\nWarning: Ran out of reviews for {app_name}. Found {len(app_results)}.")
                break

            app_results.extend(new_result)
            pbar.update(len(new_result))

    # Cap at exactly reviews_per_app in case the last batch overshot.
    app_results = app_results[:reviews_per_app]

    for review in app_results:
        review['appName'] = app_name

    all_reviews.extend(app_results)


Scraping reviews for ChatGPT...


100097it [06:59, 238.52it/s]                           


Scraping reviews for Gemini...


100097it [05:34, 299.05it/s]                           


Scraping reviews for Claude...


 33%|███▎      | 33329/100000 [01:27<02:55, 379.23it/s]

In [8]:
df_reviews = pd.DataFrame(all_reviews)
output_file = 'ai_apps_reviews_100000.csv'
df_reviews.to_csv(output_file, index=False)
print(f"Saved {len(df_reviews)} total reviews across {len(app_ids)} apps to '{output_file}'.")
df_reviews.head()

Finished
Saved 233329 total reviews across 3 apps to 'ai_apps_reviews_100000.csv'.


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,appName
0,e4fa3d7f-e865-42c4-a239-85c5dd5ea8ff,Shimran Singh,https://play-lh.googleusercontent.com/a/ACg8oc...,very good and helpful 🥹,5,0,1.2026.090,2026-04-16 09:13:22,NaN,NaT,1.2026.090,ChatGPT
1,0d8f8007-a36b-4ddd-94d9-9e0406895312,Umar Farooq,https://play-lh.googleusercontent.com/a/ACg8oc...,good,5,0,1.2026.097,2026-04-16 09:13:09,NaN,NaT,1.2026.097,ChatGPT
2,7120266f-1987-432b-a1c9-52ba91ee3ade,Ritesh bagmor Ji,https://play-lh.googleusercontent.com/a/ACg8oc...,good,5,0,1.2026.097,2026-04-16 09:12:53,NaN,NaT,1.2026.097,ChatGPT
3,07fd0c25-8ef3-491d-8006-0820e3f8ea3e,Ucha Boss,https://play-lh.googleusercontent.com/a/ACg8oc...,very very good 👍,5,0,1.2026.097,2026-04-16 09:12:34,NaN,NaT,1.2026.097,ChatGPT
4,d1e1b022-9160-4dae-9827-1bdd15039868,Odonchimeg Odonchimeg,https://play-lh.googleusercontent.com/a/ACg8oc...,This app is an absolute piece of garbage. It’s...,1,0,NaN,2026-04-16 09:12:19,NaN,NaT,NaN,ChatGPT


In [ ]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

df_reviews['Sentiment'] = df_reviews['score'].apply(classify_sentiment)
df_balanced = balance_by_min_class(df_reviews, label_col='Sentiment', random_state=42)
df_balanced.to_csv('ai_apps_reviews_BALANCED.csv', index=False)
print(f"Balanced rows: {len(df_balanced)} | counts: {df_balanced['Sentiment'].value_counts().to_dict()}")

In [5]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    words = text.split()
    return " ".join(w for w in words if w not in stop_words)

df_model = df_balanced.dropna(subset=['appName', 'content', 'Sentiment']).copy()
df_model['content'] = df_model['content'].astype(str).str.strip()
df_model = df_model[df_model['content'] != ""]

before_dedup = len(df_model)
df_model = df_model.drop_duplicates(subset=['appName', 'content'], keep='first').reset_index(drop=True)
removed_duplicates = before_dedup - len(df_model)

print(f"Rows after filtering: {before_dedup}")
print(f"Removed duplicates: {removed_duplicates}")
print(f"Rows after dedup: {len(df_model)}")

print("Cleaning text...")
df_model['clean_content'] = df_model['content'].apply(clean_text)

# Drop rows where clean_content is empty — empty strings become NaN on CSV round-trip,
# which breaks downstream TF-IDF and sanity checks.
before_empty = len(df_model)
df_model = df_model[df_model['clean_content'].str.strip() != ''].reset_index(drop=True)
print(f"Removed {before_empty - len(df_model)} rows with empty clean_content")

train_df, test_df = train_test_split(
    df_model,
    test_size=0.2,
    random_state=42,
    stratify=df_model['Sentiment'],
)

print(f"\nTrain: {len(train_df)} | Test: {len(test_df)}")

train_df.to_csv('train_data.csv', index=False)
test_df.to_csv('test_data.csv', index=False)
print("Saved train_data.csv and test_data.csv.")

Rows after filtering missing/empty fields: 30900
Removed duplicates by ['appName', 'content']: 10081
Rows after deduplication: 20819
Cleaning text (this might take a few seconds)...
Removed 296 rows with empty clean_content after stopword/punct removal
Rows kept: 20523

Training Data (80%): 16418 rows
Test Data (20%): 4105 rows
Success! Saved 'train_data.csv' and 'test_data.csv'.
